In [1]:
import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit.circuit import Parameter

from qiskit_aer import AerSimulator


In [ ]:
def uniform_over_range(num_qubits: int, M: int):
    """
    Returns a circuit that prepares a uniform superposition over |0>,|1>,...,|M-1> on num_qubits qubits.
    Uses a Hadamard layer if M is a power of 2, else uses the method of Shukla and Vedula.
    """
    if M not in range(2 ** num_qubits +1):
        print(M)
        print(num_qubits)
        raise Exception('Bad M: out of range')
    for i in range(num_qubits+1):
        if M == 2 ** i:
            print(f'M={M} a power of 2. Use Hadamard circuit.')
            circuit = QuantumCircuit(num_qubits)
            for j in range(i):
                circuit.h(j)
                
            return circuit
    
    circuit = QuantumCircuit(num_qubits)

    try:
        M_binary = np.binary_repr(M, num_qubits)
    except Exception as e:
        print(M)
        print(num_qubits)
        raise e
    M_binary = M_binary[::-1]
    ran = np.arange(len(M_binary))
    mask = [M_binary[x] == '1' for x in range(len(M_binary))]
    l = ran[mask]
    
    for i in range(1, len(l)):
        circuit.x(l[i])
    if l[0] > 0:
        for i in range(l[0]):
            circuit.h(i)

    MM = 2 ** l[0]

    circuit.ry(-2 * np.arccos(np.sqrt(MM/M)), l[1])

    for i in range(l[0], l[1]):
        circuit.ch(l[1], i, ctrl_state=0)

    for m in range(1, len(l)-1):
        circuit.cry(
            -2 * np.arccos(np.sqrt(2 ** l[m] / (M - MM) )), 
            l[m], l[m+1], ctrl_state=0
        )
        for i in range(l[m], l[m+1]):
            circuit.ch(l[m+1], i, ctrl_state=0)
        MM += 2 ** l[m]

    return circuit


def state_prep(N: int, T: int) -> QuantumCircuit:
    n = int(np.ceil(np.log2(N+1)))
    uni = uniform_over_range(n, N+1)
    circuit = QuantumCircuit(n * T)
    for t in range(T):
        circuit.append(
            uni,
            list(range(t * n, (t+1) * n))   
        )
    return circuit


def get_mixer_operator(N: int, T: int, parameter=Parameter('beta')) -> QuantumCircuit:
    # TODO: use ancillas to reduce depth of mcp?
    num_qubits = int(np.ceil(np.log2(N+1))) * T
    state_prep_circuit = state_prep(N, T)
    mixer = QuantumCircuit(num_qubits)
    mixer.append(
        state_prep_circuit.inverse(),
        range(num_qubits)
    )
    # mixer.save_statevector('after_prep')
    mixer.x(-1)
    mixer.mcp(-parameter, list(range(num_qubits - 1)), -1, ctrl_state=0)
    mixer.x(-1)
    # mixer.save_statevector('after_phase')
    mixer.append(
        state_prep_circuit,
        range(num_qubits)
    )
    # mixer.save_statevector('after_unprep')
    return mixer

In [ ]:
mixer_circuit = get_mixer_operator(3, 4)

In [ ]:
mixer_circuit.decompose(['circuit*'], reps=2).draw(fold=-1)

In [ ]:
state_prep_circuit = state_prep(4,5)
state_prep_circuit.save_statevector()
state_prep_circuit.decompose(['circuit*'], reps=2).draw(fold=-1)

In [ ]:
sim = AerSimulator()


In [ ]:
tsp = transpile(state_prep_circuit, backend=sim)

In [ ]:
res = sim.run(tsp).result()

In [ ]:
res.results[0].data

In [ ]:
sv = res.data()["statevector"]
sv = np.real_if_close(sv)

In [ ]:
sv = sv[np.abs(sv) >= 10**-4]
np.nonzero(sv)

In [ ]:
svr = sv.reshape([8]*5)

In [ ]:
svr[np.abs(svr) <= 10**-4] = 0

In [ ]:
svr[3,4,3,1,:]

In [2]:
import pickle

In [3]:
with open('/lustre/scratch127/qpg/jc59/hubo/simulation.optimisation_test_N4_W5_extra0_four0.0_six1.0.cvar0.25.p2.shots10000.initramp.d15.pkl', 'rb') as f:
    data = pickle.load(f)


In [4]:
last_samples = data['history'][-1]
last_samples = last_samples[3]
keys = list(last_samples.keys())


In [5]:
hamiltonian = data["hamiltonian"]

In [6]:
from qiskit_qaoa.utils.string_utils import evaluate_sparse_pauli_samples


In [7]:
from qiskit_qaoa.utils.gfa_utils import gfa_file_to_graph
from qiskit_qaoa.hubo.graph_to_hubo_hamiltonian import graph_to_hubo_hamiltonian


In [8]:
costs = evaluate_sparse_pauli_samples(keys, hamiltonian)

In [9]:
np.argmax(costs)

np.int64(77)

In [10]:
keys[77] # '10011111100111011001' -> 9, 15, 9, 11, 9 -> INVALID * 5

'10011111100111011001'

In [11]:
costs[77]

np.float64(62.0)

In [12]:
filepath = '/nfs/users/nfs_j/jc59/quantumwork/pangenome/data/test_N4_W5.gfa'
graph, n, N, T = gfa_file_to_graph(filepath, [2,1,1,1])


In [13]:
new_hamiltonian = graph_to_hubo_hamiltonian(graph, n, N, T, 10)

In [17]:
evaluate_sparse_pauli_samples(
    ['00011001'+'1111'*3, '10011111100111011001', '100000011000'+'1111'*2, '0000'+'0100'+'1010'+'0000'+'0110', '1110'+'1000'+'0010'+'1100'+'1000'],
    new_hamiltonian
)

array([47., 47., 33.,  0.,  0.])